In [ ]:
# %pip install -U google-cloud-aiplatform google-cloud-bigquery pandas pyarrow db-dtypes statsforecast mlforecast utilsforecast lightgbm xgboost scikit-learn

In [ ]:
from pathlib import Path

PROJECT_ID = "your-project-id"
REGION = "us-central1"
STAGING_BUCKET = "gs://your-staging-bucket"

BQ_DATASET = "your_dataset"
BQ_TABLE = "your_table"

UID_COL = "uid_col"
DATE_COL = "date_col"
TARGET_COL = "target_col"

SYS_ID_VALUE = "your_sys_id"
UID_DELIMITER = "_"

FREQ = "MS"
SEASON_LENGTH = 12
HORIZON = 12
VALIDATION_HORIZON = 12
MIN_HISTORY_FOR_FULL_AUDIT = 24

ML_LAGS = [1, 12]

MAX_SERIES = 100  # set to 0 to process all series (use for testing)

MACHINE_TYPE = "n1-standard-16"
BOOT_DISK_SIZE_GB = 200

DISPLAY_NAME = f"nixtla-custom-job-{SYS_ID_VALUE.lower().replace('_', '-')}"
CONTAINER_URI = "us-docker.pkg.dev/vertex-ai/training/sklearn-cpu.1-6:latest"
VERTEX_SERVICE_ACCOUNT = "your-vertex-sa@your-project-id.iam.gserviceaccount.com"

OUTPUT_DATASET = "Nixtla_final_forecast_results_custom_job"
FORECAST_TABLE = "nixtla_final_forecast_results"
METRICS_TABLE = "customer_model_performance_metrics"
CHAMPIONS_TABLE = "best_model_metadata"


LOCAL_SCRIPT_PATH = Path("train_nixtla_custom_job.py")


In [ ]:
# Create output BQ dataset if it does not already exist
from google.cloud import bigquery as _bq

_bq_client = _bq.Client(project=PROJECT_ID)
_dataset_ref = _bq.Dataset(f"{PROJECT_ID}.{OUTPUT_DATASET}")
_dataset_ref.location = REGION
_bq_client.create_dataset(_dataset_ref, exists_ok=True)
print(f"Dataset ready: {PROJECT_ID}.{OUTPUT_DATASET}")

In [ ]:
training_script = r'''
import argparse
import json
import os
import re
import ray

import numpy as np
import pandas as pd
from google.cloud import bigquery

from statsforecast import StatsForecast
from statsforecast.models import (
    AutoARIMA,
    AutoETS,
    AutoTheta,
    MSTL,
    ARCH,
    GARCH,
    ADIDA,
    CrostonOptimized,
    Naive,
    SeasonalNaive,
)
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--project_id", required=True)
    parser.add_argument("--region", required=True)
    parser.add_argument("--bq_dataset", required=True)
    parser.add_argument("--bq_table", required=True)
    parser.add_argument("--uid_col", required=True)
    parser.add_argument("--date_col", required=True)
    parser.add_argument("--target_col", required=True)
    parser.add_argument("--sys_id_value", required=True)
    parser.add_argument("--uid_delimiter", required=True)
    parser.add_argument("--freq", default="MS")
    parser.add_argument("--season_length", type=int, default=12)
    parser.add_argument("--horizon", type=int, default=12)
    parser.add_argument("--validation_horizon", type=int, default=12)
    parser.add_argument("--min_history_for_full_audit", type=int, default=24)
    parser.add_argument("--max_series", type=int, default=0)
    parser.add_argument("--ml_lags", default="1,12")
    parser.add_argument("--output_dataset", required=True)
    parser.add_argument("--forecast_table", required=True)
    parser.add_argument("--metrics_table", required=True)
    parser.add_argument("--champions_table", required=True)
    return parser.parse_args()


def utc_now() -> pd.Timestamp:
    return pd.Timestamp.now(tz="UTC").tz_localize(None)


def build_query(project_id: str, dataset: str, table: str) -> str:
    return f"""
    WITH base AS (
      SELECT
        CAST({{uid_col}} AS STRING) AS unique_id,
        DATE({{date_col}}) AS ds,
        CAST({{target_col}} AS FLOAT64) AS y
      FROM `{project_id}.{dataset}.{table}`
      WHERE {{uid_col}} IS NOT NULL
        AND {{date_col}} IS NOT NULL
        AND {{target_col}} IS NOT NULL
    ),
    parsed AS (
      SELECT
        SPLIT(unique_id, @uid_delimiter)[SAFE_OFFSET(0)] AS sys_id,
        unique_id,
        ds,
        y
      FROM base
    )
    SELECT
      unique_id,
      ds,
      y,
      sys_id
    FROM parsed
    WHERE sys_id = @sys_id_value
    ORDER BY unique_id, ds
    """


def read_data(args) -> pd.DataFrame:
    client = bigquery.Client(project=args.project_id)
    query = build_query(args.project_id, args.bq_dataset, args.bq_table).format(
        uid_col=args.uid_col,
        date_col=args.date_col,
        target_col=args.target_col,
    )
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("uid_delimiter", "STRING", args.uid_delimiter),
            bigquery.ScalarQueryParameter("sys_id_value", "STRING", args.sys_id_value),
        ]
    )
    df = client.query(query, job_config=job_config).to_dataframe(create_bqstorage_client=True)
    if df.empty:
        raise ValueError(f"No rows returned for sys_id={args.sys_id_value}")
    df["ds"] = pd.to_datetime(df["ds"])
    if args.max_series and args.max_series > 0:
        uids = df["unique_id"].unique()[:args.max_series]
        df = df[df["unique_id"].isin(uids)]
        print(f"[max_series] Limiting to {len(uids)} series for testing.")
    return df


def aggregate_monthly(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["ds"] = df["ds"].dt.to_period("M").dt.to_timestamp()
    out = (
        df.groupby(["unique_id", "ds"], as_index=False)["y"]
        .sum()
        .sort_values(["unique_id", "ds"])
        .reset_index(drop=True)
    )
    return out


def split_ids_by_history(df: pd.DataFrame, min_history: int):
    counts = df.groupby("unique_id").size()
    eligible_ids = counts[counts >= min_history].index.tolist()
    fallback_ids = counts[counts < min_history].index.tolist()
    return eligible_ids, fallback_ids


def split_train_valid(df: pd.DataFrame, validation_horizon: int):
    train_parts = []
    valid_parts = []
    for _, grp in df.groupby("unique_id", sort=False):
        grp = grp.sort_values("ds").reset_index(drop=True)
        if len(grp) <= validation_horizon:
            continue
        train_parts.append(grp.iloc[:-validation_horizon].copy())
        valid_parts.append(grp.iloc[-validation_horizon:].copy())
    if not train_parts or not valid_parts:
        raise ValueError("Not enough history to build train/validation split.")
    train_df = pd.concat(train_parts, ignore_index=True)
    valid_df = pd.concat(valid_parts, ignore_index=True)
    return train_df, valid_df


def build_stats_models(season_length: int):
    return [
        AutoARIMA(season_length=season_length),
        AutoETS(season_length=season_length),
        AutoTheta(season_length=season_length),
        MSTL(season_length=[season_length], trend_forecaster=AutoARIMA(season_length=1)),
        ARCH(),
        GARCH(),
        ADIDA(),
        CrostonOptimized(),
        Naive(),
        SeasonalNaive(season_length=season_length),
    ]


@ray.remote
def _stats_audit_uid_remote(uid_records: list, horizon: int, freq: str, season_length: int) -> list:
    try:
        df = pd.DataFrame(uid_records)
        df["ds"] = pd.to_datetime(df["ds"])
        df = df.sort_values("ds").reset_index(drop=True)
        sf = StatsForecast(
            models=build_stats_models(season_length),
            freq=freq,
            n_jobs=1,
        )
        preds = sf.forecast(df=df, h=horizon)
        preds["ds"] = pd.to_datetime(preds["ds"])
        return preds.to_dict(orient="records")
    except Exception as e:
        uid = uid_records[0].get("unique_id") if uid_records else "<unknown>"
        print(f"[stats_audit_error] uid={uid} error={e!r}")
        return []


def run_stats_audit(train_df: pd.DataFrame, horizon: int, freq: str, season_length: int) -> pd.DataFrame:
    futures = []
    for _, grp in train_df.groupby("unique_id", sort=False):
        futures.append(
            _stats_audit_uid_remote.remote(
                grp.to_dict(orient="records"),
                horizon,
                freq,
                season_length,
            )
        )
    results = ray.get(futures)
    rows = [row for batch in results for row in batch]
    if not rows:
        return pd.DataFrame(columns=["unique_id", "ds"])
    out = pd.DataFrame(rows)
    out["ds"] = pd.to_datetime(out["ds"])
    return out.sort_values(["unique_id", "ds"]).reset_index(drop=True)


def build_mlforecast(lags: list, freq: str, num_threads: int = -1):
    models = {
        "XGBoost": XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            objective="reg:squarederror",
            n_jobs=1,
        ),
        "LightGBM": LGBMRegressor(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=64,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            verbosity=-1,
            n_jobs=1,
        ),
    }
    lag_transforms = {
        lags[0]: [RollingMean(window_size=3), RollingMean(window_size=6), RollingMean(window_size=12)],
        lags[-1]: [RollingMean(window_size=3)],
    }
    return MLForecast(
        models=models,
        freq=freq,
        lags=lags,
        lag_transforms=lag_transforms,
        date_features=["month", "quarter", "year"],
        num_threads=num_threads,
    )


def run_ml_audit(train_df: pd.DataFrame, horizon: int, lags: list, freq: str, num_threads: int = -1) -> pd.DataFrame:
    fcst = build_mlforecast(lags, freq, num_threads=num_threads)
    fcst.fit(
        train_df,
        id_col="unique_id",
        time_col="ds",
        target_col="y",
        static_features=[],
    )
    preds = fcst.predict(horizon)
    preds["ds"] = pd.to_datetime(preds["ds"])
    return preds


def evaluate_predictions(valid_df: pd.DataFrame, preds: pd.DataFrame) -> pd.DataFrame:
    merged = valid_df.merge(preds, on=["unique_id", "ds"], how="inner")
    model_cols = [c for c in merged.columns if c not in {"unique_id", "ds", "y"}]
    if not model_cols:
        raise ValueError("No forecast columns found for evaluation.")
    metric_df = evaluate(
        merged,
        metrics=[rmse],
        models=model_cols,
        id_col="unique_id",
        target_col="y",
    )
    metric_df = metric_df[metric_df["metric"] == "rmse"].drop(columns=["metric"])
    metric_long = metric_df.melt(
        id_vars=["unique_id"],
        var_name="model",
        value_name="rmse",
    ).sort_values(["unique_id", "rmse", "model"]).reset_index(drop=True)
    return metric_long


def build_champions(stats_metrics: pd.DataFrame, ml_metrics: pd.DataFrame, fallback_ids, sys_id_value: str) -> pd.DataFrame:
    all_metrics = pd.concat([stats_metrics, ml_metrics], ignore_index=True)
    best = (
        all_metrics.sort_values(["unique_id", "rmse", "model"])
        .groupby("unique_id", as_index=False)
        .first()
        .rename(columns={"model": "best_model_name"})
    )
    best["sys_id"] = sys_id_value

    if fallback_ids:
        fallback_df = pd.DataFrame({
            "unique_id": fallback_ids,
            "best_model_name": "Naive_Fallback",
            "rmse": np.nan,
            "sys_id": sys_id_value,
        })
        best = pd.concat([best, fallback_df], ignore_index=True)

    return best[["sys_id", "unique_id", "best_model_name", "rmse"]].sort_values(["unique_id"]).reset_index(drop=True)


def fit_single_stats_model(model_name: str, uid_df: pd.DataFrame, horizon: int, freq: str, season_length: int) -> pd.DataFrame:
    model_map = {
        "AutoARIMA": AutoARIMA(season_length=season_length),
        "AutoETS": AutoETS(season_length=season_length),
        "AutoTheta": AutoTheta(season_length=season_length),
        "MSTL": MSTL(season_length=[season_length], trend_forecaster=AutoARIMA(season_length=1)),
        "ARCH": ARCH(),
        "GARCH": GARCH(),
        "ADIDA": ADIDA(),
        "CrostonOptimized": CrostonOptimized(),
        "Naive": Naive(),
        "SeasonalNaive": SeasonalNaive(season_length=season_length),
        "Naive_Fallback": Naive(),
    }
    sf = StatsForecast(models=[model_map[model_name]], freq=freq, n_jobs=1)
    pred = sf.forecast(df=uid_df, h=horizon)
    pred["ds"] = pd.to_datetime(pred["ds"])
    forecast_col = [c for c in pred.columns if c not in {"unique_id", "ds"}][0]
    return pred.rename(columns={forecast_col: "Forecast"})[["unique_id", "ds", "Forecast"]]


def fit_single_ml_model(model_name: str, uid_df: pd.DataFrame, horizon: int, lags: list, freq: str) -> pd.DataFrame:
    if model_name == "XGBoost":
        models = {
            "fcst_temp": XGBRegressor(
                n_estimators=400,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=42,
                objective="reg:squarederror",
                n_jobs=1,
            )
        }
    elif model_name == "LightGBM":
        models = {
            "fcst_temp": LGBMRegressor(
                n_estimators=400,
                learning_rate=0.05,
                num_leaves=64,
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=42,
                verbosity=-1,
                n_jobs=1,
            )
        }
    else:
        raise ValueError(f"Unsupported ML model: {model_name}")

    fcst = MLForecast(
        models=models,
        freq=freq,
        lags=lags,
        lag_transforms={
            lags[0]: [RollingMean(window_size=3), RollingMean(window_size=6), RollingMean(window_size=12)],
            lags[-1]: [RollingMean(window_size=3)],
        },
        date_features=["month", "quarter", "year"],
        num_threads=1,
    )
    fcst.fit(
        uid_df,
        id_col="unique_id",
        time_col="ds",
        target_col="y",
        static_features=[],
    )
    pred = fcst.predict(horizon)
    pred["ds"] = pd.to_datetime(pred["ds"])
    return pred.rename(columns={"fcst_temp": "Forecast"})[["unique_id", "ds", "Forecast"]]


@ray.remote
def _forecast_uid_remote(uid: str, grp_records: list, best_model: str, horizon: int, freq: str, season_length: int, sys_id_value: str, lags: list) -> list:
    ml_models = {"XGBoost", "LightGBM"}
    try:
        grp = pd.DataFrame(grp_records)
        grp["ds"] = pd.to_datetime(grp["ds"])
        grp = grp.sort_values("ds").reset_index(drop=True)

        if best_model in ml_models:
            pred = fit_single_ml_model(best_model, grp, horizon, lags, freq)
        else:
            pred = fit_single_stats_model(best_model, grp, horizon, freq, season_length)

        pred["sys_id"] = sys_id_value
        pred["Best_model_name"] = best_model
        return pred[["sys_id", "unique_id", "ds", "Forecast", "Best_model_name"]].to_dict(orient="records")
    except Exception as e:
        print(f"[forecast_error] uid={uid} model={best_model} error={e!r}")
        return []


def refit_and_forecast(full_df: pd.DataFrame, champions_df: pd.DataFrame, horizon: int, freq: str, season_length: int, sys_id_value: str, lags: list) -> pd.DataFrame:
    champ_map = dict(zip(champions_df["unique_id"], champions_df["best_model_name"]))
    futures = []
    for uid, grp in full_df.groupby("unique_id", sort=False):
        best_model = champ_map.get(uid)
        if best_model is None:
            continue
        futures.append(
            _forecast_uid_remote.remote(
                uid,
                grp.to_dict(orient="records"),
                best_model,
                horizon,
                freq,
                season_length,
                sys_id_value,
                lags,
            )
        )

    results = ray.get(futures)
    rows = [row for batch in results for row in batch]
    if not rows:
        return pd.DataFrame(columns=["sys_id", "unique_id", "ds", "Forecast", "Best_model_name", "created_at"])
    out = pd.DataFrame(rows)
    out["ds"] = pd.to_datetime(out["ds"])
    out["created_at"] = utc_now()
    return out.sort_values(["unique_id", "ds"]).reset_index(drop=True)


def build_metrics_output(stats_metrics: pd.DataFrame, ml_metrics: pd.DataFrame, champions_df: pd.DataFrame) -> pd.DataFrame:
    all_metrics = pd.concat([stats_metrics, ml_metrics], ignore_index=True)
    if all_metrics.empty:
        return pd.DataFrame()
    wide = all_metrics.pivot_table(index="unique_id", columns="model", values="rmse", aggfunc="first")
    wide.columns = [re.sub(r"[^a-zA-Z0-9_]", "_", f"{col}_RMSE") for col in wide.columns]
    wide = wide.reset_index()
    best = champions_df[["unique_id", "best_model_name", "rmse"]].rename(
        columns={"best_model_name": "Best_model_name", "rmse": "Best_model_RMSE"}
    )
    wide = wide.merge(best, on="unique_id", how="left")
    wide["created_at"] = utc_now()
    return wide


def write_bq(df: pd.DataFrame, table_fqn: str, project_id: str, write_disposition: str = "WRITE_APPEND"):
    client = bigquery.Client(project=project_id)
    df = df.drop(columns=["sys_id"], errors="ignore")
    job_config = bigquery.LoadJobConfig(write_disposition=write_disposition)
    if write_disposition == "WRITE_APPEND":
        job_config.schema_update_options = [bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION]
    job = client.load_table_from_dataframe(df, table_fqn, job_config=job_config)
    job.result()


def main():
    args = parse_args()
    ml_lags = [int(x) for x in args.ml_lags.split(",")]

    n_cpu = os.cpu_count() or 1
    print(f"[cpu] cpu_count={n_cpu} (stats audit via Ray, ml audit global with all cores)")

    ray.init(ignore_reinit_error=True, logging_level="WARNING", include_dashboard=False)

    raw_df = read_data(args)
    full_df = aggregate_monthly(raw_df)

    eligible_ids, fallback_ids = split_ids_by_history(full_df, args.min_history_for_full_audit)
    eligible_df = full_df[full_df["unique_id"].isin(eligible_ids)].copy()

    stats_metrics = pd.DataFrame(columns=["unique_id", "model", "rmse"])
    ml_metrics = pd.DataFrame(columns=["unique_id", "model", "rmse"])

    if not eligible_df.empty:
        train_df, valid_df = split_train_valid(eligible_df, args.validation_horizon)

        # Stats audit: per-uid Ray fanout (each task fits all 10 stats models for one series).
        print(f"[audit] running stats audit via Ray ({eligible_df['unique_id'].nunique()} series x 10 models)")
        stats_preds = run_stats_audit(
            train_df=train_df,
            horizon=args.validation_horizon,
            freq=args.freq,
            season_length=args.season_length,
        )

        # ML audit: global fit across all series, internally multi-threaded with all cores.
        print("[audit] running ml audit (XGBoost + LightGBM, global fit)")
        ml_preds = run_ml_audit(
            train_df=train_df,
            horizon=args.validation_horizon,
            lags=ml_lags,
            freq=args.freq,
            num_threads=n_cpu,
        )

        stats_metrics = evaluate_predictions(valid_df, stats_preds)
        ml_metrics = evaluate_predictions(valid_df, ml_preds)

    champions_df = build_champions(
        stats_metrics=stats_metrics,
        ml_metrics=ml_metrics,
        fallback_ids=fallback_ids,
        sys_id_value=args.sys_id_value,
    )
    metrics_df = build_metrics_output(stats_metrics, ml_metrics, champions_df)

    forecast_df = refit_and_forecast(
        full_df=full_df,
        champions_df=champions_df,
        horizon=args.horizon,
        freq=args.freq,
        season_length=args.season_length,
        sys_id_value=args.sys_id_value,
        lags=ml_lags,
    )

    forecast_table_fqn = f"{args.project_id}.{args.output_dataset}.{args.forecast_table}"
    metrics_table_fqn = f"{args.project_id}.{args.output_dataset}.{args.metrics_table}"
    champions_table_fqn = f"{args.project_id}.{args.output_dataset}.{args.champions_table}"

    if not metrics_df.empty:
        write_bq(metrics_df, metrics_table_fqn, args.project_id, write_disposition="WRITE_TRUNCATE")
    write_bq(champions_df, champions_table_fqn, args.project_id)
    write_bq(forecast_df, forecast_table_fqn, args.project_id)

    print(json.dumps({
        "sys_id": args.sys_id_value,
        "max_series": args.max_series,
        "series_count": int(full_df["unique_id"].nunique()),
        "eligible_ids_for_full_audit": len(eligible_ids),
        "fallback_ids": len(fallback_ids),
        "metrics_rows": int(len(metrics_df)),
        "champion_rows": int(len(champions_df)),
        "forecast_rows": int(len(forecast_df)),
    }, indent=2, default=str))


if __name__ == "__main__":
    main()
'''

print(LOCAL_SCRIPT_PATH.resolve())
LOCAL_SCRIPT_PATH.write_text(training_script)


In [ ]:
requirements = [
    "google-cloud-bigquery>=3.30.0",
    "google-cloud-aiplatform>=1.144.0",
    "pandas>=2.2.0",
    "numpy>=1.26.0",
    "pyarrow>=15.0.0",
    "db-dtypes>=1.2.0",
    "statsforecast>=2.0.0",
    "mlforecast>=1.0.2",
    "utilsforecast>=0.2.12",
    "lightgbm>=4.0.0",
    "xgboost>=2.0.0",
    "scikit-learn>=1.4.0",
    "python-json-logger>=2.0.0",
    "ray>=2.9.0",
]
requirements

In [ ]:
from google.cloud import aiplatform

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=STAGING_BUCKET,
)

In [ ]:
job = aiplatform.CustomJob.from_local_script(
    display_name=DISPLAY_NAME,
    script_path=str(LOCAL_SCRIPT_PATH),
    container_uri=CONTAINER_URI,
    requirements=requirements,
    service_account=VERTEX_SERVICE_ACCOUNT,
    machine_type=MACHINE_TYPE,
    boot_disk_size_gb=BOOT_DISK_SIZE_GB,
    replica_count=1,
    args=[
        "--project_id", PROJECT_ID,
        "--region", REGION,
        "--bq_dataset", BQ_DATASET,
        "--bq_table", BQ_TABLE,
        "--uid_col", UID_COL,
        "--date_col", DATE_COL,
        "--target_col", TARGET_COL,
        "--sys_id_value", SYS_ID_VALUE,
        "--uid_delimiter", UID_DELIMITER,
        "--freq", FREQ,
        "--season_length", str(SEASON_LENGTH),
        "--horizon", str(HORIZON),
        "--validation_horizon", str(VALIDATION_HORIZON),
        "--min_history_for_full_audit", str(MIN_HISTORY_FOR_FULL_AUDIT),
        "--max_series", str(MAX_SERIES),
        "--ml_lags", ",".join(str(l) for l in ML_LAGS),
        "--output_dataset", OUTPUT_DATASET,
        "--forecast_table", FORECAST_TABLE,
        "--metrics_table", METRICS_TABLE,
        "--champions_table", CHAMPIONS_TABLE,
    ],
)
job


In [ ]:
job.run(sync=True)

In [ ]:
job.resource_name